# Week 4: train on balanced RES/NRES pixels

Pure Track-A cells only, count-balanced per metro (`n_RES = n_NRES` via NRES undersample, seed=42).

| | |
|---|---|
| Data | `datasets/week4_pixel_gt_balanced/` |
| Train | LORO Ridge (3 fracs) + LR (binary), no SMOTE |
| Outputs | `outputs/week4_pixel_balanced_train/` |

Rebuild:
```bash
.venv/bin/python scripts/build_week4_pixel_gt_balanced.py
.venv/bin/python scripts/train_week4_pixel_balanced_loro.py
```


In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / "datasets" / "week4_pixel_gt_balanced" / "manifest.json").exists():
        ROOT = p
        break

BAL = ROOT / "datasets" / "week4_pixel_gt_balanced"
OUT = ROOT / "outputs" / "week4_pixel_balanced_train"
bal_man = json.loads((BAL / "manifest.json").read_text())
train_sum = json.loads((OUT / "summary.json").read_text())
df = pd.read_parquet(BAL / "pixels.parquet", columns=["region", "stratum", "y", "frac_res", "frac_nres", "frac_empty"])
metrics = pd.read_csv(OUT / "loro_metrics.csv")

print(
    f"balanced n={bal_man['n_pixels']:,}  "
    f"RES={bal_man['n_residential']:,}  NRES={bal_man['n_non_residential']:,}"
)
display(pd.DataFrame(bal_man["counts_by_region"]).T)
print(
    f"OOF mae_mean={train_sum['oof_mae_mean']:.4f}  "
    f"macroF1={train_sum['oof_macro_f1']:.4f}  "
    f"LORO F1={train_sum['loro_macro_f1_mean']:.3f}±{train_sum['loro_macro_f1_std']:.3f}"
)
display(metrics[["region", "n_test", "n_train_res", "n_train_nres", "mae_mean", "macro_f1"]].round(3))


balanced n=59,596  RES=29,798  NRES=29,798


,residential,non_residential
Atlanta_SE,4670,4670
Boston_NE,3344,3344
Chicago_MW,3068,3068
Denver_W,2751,2751
Houston_S,4132,4132
LA_W,1544,1544
NYC_NE,2767,2767
Phoenix_SW,2863,2863
SF_W,1411,1411
Seattle_NW,3248,3248


OOF mae_mean=0.2807  macroF1=0.7043  LORO F1=0.704±0.022


,region,n_test,n_train_res,n_train_nres,mae_mean,macro_f1
0,Atlanta_SE,9340,25128,25128,0.293,0.673
1,Boston_NE,6688,26454,26454,0.267,0.675
2,Chicago_MW,6136,26730,26730,0.262,0.740
3,Denver_W,5502,27047,27047,0.271,0.696
4,Houston_S,8264,25666,25666,0.301,0.681
5,LA_W,3088,28254,28254,0.258,0.735
6,NYC_NE,5534,27031,27031,0.285,0.702
7,Phoenix_SW,5726,26935,26935,0.284,0.706
8,SF_W,2822,28387,28387,0.292,0.720
9,Seattle_NW,6496,26550,26550,0.276,0.714


## Balance check

Before: NRES much larger than RES. After: equal counts in every metro.

![Imbalanced](../../datasets/week4_pixel_gt_balanced/figures/imbalanced_counts_by_region.png)

![Balanced](../../datasets/week4_pixel_gt_balanced/figures/balanced_counts_by_region.png)


## Training results (LORO)

Binary LR macro-F1 0.704 OOF (LORO mean 0.704 ± 0.022). Fraction mean MAE 0.281.
Train folds stay balanced (RES=NRES) without SMOTE.

![F1 by region](../../outputs/week4_pixel_balanced_train/figures/loro_f1_by_region.png)

![MAE by region](../../outputs/week4_pixel_balanced_train/figures/loro_mae_by_region.png)

![True vs pred](../../outputs/week4_pixel_balanced_train/figures/oof_true_vs_pred.png)

![Confusion](../../outputs/week4_pixel_balanced_train/figures/oof_confusion.png)


## Eval on balanced GT (vs Week 3 models)

Rescore join is on the balanced `pixel_id` set only.

```bash
.venv/bin/python scripts/eval_week4_pixel.py \
  --run-id balanced_rescore_v1 \
  --gt datasets/week4_pixel_gt_balanced/pixels.parquet
```


In [2]:
EVAL = ROOT / "outputs" / "week4_pixel_eval" / "balanced_rescore_v1"
if (EVAL / "summary_metrics.csv").exists():
    board = pd.read_csv(EVAL / "summary_metrics.csv")
    cols = [
        "model", "n_joined",
        "oof_mae_frac_res", "oof_mae_frac_nres", "oof_mae_frac_empty",
        "oof_mae_mean", "loro_mae_mean_mean",
    ]
    display(board[cols].sort_values("oof_mae_mean").round(4))
else:
    print("Run eval first (see command above).")


,model,n_joined,oof_mae_frac_res,oof_mae_frac_nres,oof_mae_frac_empty,oof_mae_mean,loro_mae_mean_mean
0,pixel_3class_reg,59596,0.2428,0.2862,0.3036,0.2776,0.2754
1,balanced_train,59596,0.2532,0.2904,0.2949,0.2795,0.2781
2,neighborhood_k3_reg,58753,0.2210,0.2997,0.3876,0.3027,0.2991
3,pixel_ae,59596,0.4171,0.3337,0.4530,0.4013,0.4022
4,fraction_regression,59596,0.4142,0.3387,0.4530,0.4020,0.4031


![MAE ladder](../../outputs/week4_pixel_eval/balanced_rescore_v1/figures/mae_ladder.png)

![Per-fraction MAE](../../outputs/week4_pixel_eval/balanced_rescore_v1/figures/mae_by_fraction.png)

![By region](../../outputs/week4_pixel_eval/balanced_rescore_v1/figures/mae_by_region.png)
